# Aegis - Phase 7: StrongREJECT End-to-End ASR

Field-standard defense evaluation (Souly et al., "A StrongREJECT for Empty Jailbreaks", arXiv:2402.10260): measure whether an attack actually makes a victim LLM emit specific harmful content, judged by the StrongREJECT evaluator, comparing the victim ALONE against the victim placed behind Aegis - and do it per obfuscation channel, so the encoding axis Aegis targets is measured directly.

**Setup:** Accelerator -> GPU T4. Settings -> Internet: ON. Add Kaggle Secrets `HF_TOKEN` and `WANDB_API_KEY`. Then Run All.

In [ ]:
import sys, os, glob, subprocess
REPO_URL = "https://github.com/g25ait2149/aegis.git"     # same repo as the other notebooks
DEST = "/kaggle/working/aegis_src"
if os.path.isdir(os.path.join(DEST, ".git")):
    subprocess.run(["git", "-C", DEST, "pull", "--ff-only"], check=False)   # already cloned -> update
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, DEST], check=False)
hits = glob.glob(DEST + "/**/aegis/__init__.py", recursive=True)
root = os.path.dirname(os.path.dirname(hits[0])) if hits else DEST
sys.path.insert(0, root)
for m in [m for m in sys.modules if m == "aegis" or m.startswith(("aegis.", "eval"))]:
    del sys.modules[m]
print("aegis repo at:", root)

## Secrets
Add as **Kaggle Secrets** (Add-ons -> Secrets):
- **`HF_TOKEN`** -> fits Aegis on the training corpus and loads the victim / judge models.
- **`WANDB_API_KEY`** -> logs the ASR table to **Weights & Biases** (project `aegis-llm-defense`).

In [ ]:
!pip -q install strong_reject accelerate

## 1. End-to-end ASR: victim alone vs victim behind Aegis

`run_strongreject` loads the StrongREJECT forbidden-prompt set, generates a victim response once per prompt, then applies Aegis as a filter on top (`scan` -> block, `guard_turn` -> block). ASR is the mean StrongREJECT judge score (lower is safer); because Aegis only blocks, its ASR can never exceed the undefended ASR. Each obfuscation channel is its own row, so `char_spacing` (the current red-team gap) and `base64` etc. are visible side by side.

In [ ]:
from eval.strongreject_eval import run_strongreject

rows = run_strongreject(
    victim_id="Qwen/Qwen2.5-1.5B-Instruct",   # small safety-tuned victim, fits a T4
    n=60,
    attacks=("identity", "base64", "char_spacing", "homoglyph", "zero_width", "roleplay_wrap"),
    judge_name="strongreject",                 # the real fine-tuned StrongREJECT judge
    wandb_log=True,
)
rows

## 2. Optional: the full cascade with the L2 guard

By default this evaluates L0-L1 plus the L4 output gate. To score the full cascade, build an `Aegis` with the fine-tuned L2 guard attached (see Phase 3 for loading it) and pass it in:

```python
# ag = Aegis(...).attach_guard(tuned_guard)   # from P3
# run_strongreject(victim_id="Qwen/Qwen2.5-1.5B-Instruct", aegis=ag, judge_name="strongreject")
```

## Next (Arc 1)
- Item 2: over-refusal on XSTest, alongside the in-house FRR.
- Item 3: Qwen3Guard-0.6B as a modern guard baseline.
- Item 4: AgentDojo (task utility and attack-success-rate jointly).